In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/nlp-a3-dataset/shakespear_dev.txt
/kaggle/input/nlp-a3-dataset/shakespear_train.txt


In [2]:
# # task1_word_level.py

# import torch
# import torch.nn as nn
# from torch.nn import functional as F
# import math
# import time
# import os
# from collections import Counter
# from tqdm import tqdm
# import matplotlib.pyplot as plt
# from torch.utils.data import Dataset, DataLoader
# import numpy as np
# import re # For basic word splitting

# # --- Configuration ---
# # Kaggle Environment Check
# IS_KAGGLE = os.path.exists('/kaggle/input')
# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {DEVICE}")

# # Hyperparameters (Adjust for word-level, MAX_LEN is crucial)
# BATCH_SIZE = 32        # Reduce batch size if memory becomes an issue
# MAX_LEN = 64           # Max sequence length (words) - sentences will be padded/truncated
# MAX_ITERS = 10000      # Adjust as needed
# EVAL_INTERVAL = 250    # How often to evaluate and print progress
# LEARNING_RATE = 1e-4   # Often lower LR is needed for word models
# EVAL_ITERS = 100       # Batches used for loss estimation (less needed maybe)
# N_EMBD = 256           # Embedding dimension
# N_HEAD = 4             # Number of attention heads
# N_LAYER = 4            # Number of transformer blocks
# DROPOUT = 0.1          # Regularization
# WEIGHT_DECAY = 0.01    # Regularization

# # --- Learning Rate Schedule Parameters (Optional but Recommended) ---
# WARMUP_ITERS = 100
# LR_DECAY_ITERS = MAX_ITERS
# MIN_LR = 1e-5

# # Data Paths
# # Assuming files are in the same directory or /kaggle/input/
# BASE_DIR = '/kaggle/input/nlp-a3-dataset/' if IS_KAGGLE else './'
# TRAIN_FILE = os.path.join(BASE_DIR, 'shakespear_train.txt')
# DEV_FILE = os.path.join(BASE_DIR, 'shakespear_dev.txt')
# TEST_FILE_DEMO = 'shakespear_test.txt' # Expected name for demo test file

# # Output Paths
# MODEL_SAVE_PATH = 'task1_transformer_word_level.pth'
# PLOT_SAVE_PATH = 'task1_loss_lr_plot_word_level.png'

# # Ensure N_EMBD is divisible by N_HEAD
# assert N_EMBD % N_HEAD == 0

# # Seed for reproducibility
# torch.manual_seed(1337)
# if torch.cuda.is_available():
#     torch.cuda.manual_seed(1337)

# # --- Special Tokens ---
# PAD_TOKEN = "<PAD>"
# UNK_TOKEN = "<UNK>"
# START_TOKEN = "<START>"
# STOP_TOKEN = "<STOP>"
# special_tokens = [PAD_TOKEN, UNK_TOKEN, START_TOKEN, STOP_TOKEN]

# # --- Word Tokenization and Preprocessing ---

# def simple_word_tokenize(text):
#     """Basic word tokenizer: lowercase, split by space/punctuation."""
#     text = text.lower()
#     # Keep basic punctuation attached to words for simplicity, split others
#     words = re.findall(r"[\w']+|[.,!?;:]", text)
#     return words

# def build_vocabulary(all_lines, min_freq=2):
#     """Builds word vocabulary from tokenized lines."""
#     print("Building word vocabulary...")
#     token_counts = Counter()
#     for line in tqdm(all_lines, desc="Counting words"):
#         token_counts.update(simple_word_tokenize(line))

#     vocab = special_tokens[:] # Start with special tokens
#     for token, count in token_counts.items():
#         if count >= min_freq:
#             vocab.append(token)
#     print(f"Vocabulary size: {len(vocab)} (min_freq={min_freq})")
#     print(f"Sample vocab: {vocab[:10]} ... {vocab[-10:]}")

#     tokenizer = {token: i for i, token in enumerate(vocab)}
#     tokenizer_inv = {i: token for token, i in tokenizer.items()}

#     # Ensure special tokens are mapped correctly
#     for st in special_tokens:
#         if st not in tokenizer:
#             print(f"FATAL: Special token '{st}' missing from tokenizer!")
#             raise ValueError("Special token error")

#     return tokenizer, tokenizer_inv, len(vocab)

# def tokenize_line(line, tokenizer, max_len, add_start_stop=True):
#     """Tokenizes a single line, adds special tokens, handles UNK, and pads/truncates."""
#     words = simple_word_tokenize(line)
#     tokens = []
#     if add_start_stop:
#         tokens.append(tokenizer[START_TOKEN])

#     for word in words:
#         tokens.append(tokenizer.get(word, tokenizer[UNK_TOKEN]))

#     if add_start_stop:
#         tokens.append(tokenizer[STOP_TOKEN])

#     # Truncate if necessary (keeping space for start/stop if added)
#     tokens = tokens[:max_len]

#     # Pad if necessary
#     padding_needed = max_len - len(tokens)
#     if padding_needed > 0:
#         tokens.extend([tokenizer[PAD_TOKEN]] * padding_needed)

#     return tokens

# class ShakespeareWordDataset(Dataset):
#     def __init__(self, lines, tokenizer, max_len):
#         self.tokenizer = tokenizer
#         self.max_len = max_len # Max length INCLUDING start/stop tokens
#         self.pad_id = tokenizer[PAD_TOKEN]
#         print(f"Tokenizing {len(lines)} lines for dataset...")
#         self.data = []
#         for line in tqdm(lines, desc="Tokenizing lines"):
#             if not line.strip(): continue # Skip empty lines
#             # +1 because input is token[0]..token[n-1], target is token[1]..token[n]
#             # We need sequences long enough to create a target
#             token_ids = tokenize_line(line, tokenizer, max_len + 1, add_start_stop=True)
#             # Ensure we have at least START and one word token to form a pair
#             if len(token_ids) > 1 and token_ids[0] == tokenizer[START_TOKEN]:
#                  # Check if sequence contains non-pad tokens besides START/STOP
#                 non_pad_count = sum(1 for tid in token_ids if tid != self.pad_id)
#                 if non_pad_count > 2: # Need at least START + word + STOP (or another word)
#                     self.data.append(torch.tensor(token_ids, dtype=torch.long))
#                 # else: print(f"Skipping short/empty line: {line.strip()}") # Debug
#             # else: print(f"Skipping very short line: {line.strip()}") # Debug

#         print(f"Created dataset with {len(self.data)} sequences.")
#         if not self.data: print("Warning: Dataset is empty!")


#     def __len__(self):
#         return len(self.data)

#     def __getitem__(self, idx):
#         full_seq = self.data[idx]
#         # Input: <START> w1 w2 ... wn <STOP> <PAD>...<PAD>
#         # Target: w1 w2 ... wn <STOP> <PAD>...<PAD> <PAD>
#         # Both should be length max_len
#         x = full_seq[:-1] # Length max_len
#         y = full_seq[1:]  # Length max_len
#         # Sanity check lengths
#         assert x.shape[0] == self.max_len, f"Input shape wrong: {x.shape[0]} vs {self.max_len}"
#         assert y.shape[0] == self.max_len, f"Target shape wrong: {y.shape[0]} vs {self.max_len}"
#         return x, y

# def load_and_preprocess_data(train_file, dev_file, test_file, max_len, batch_size):
#     """Loads data, builds vocab, creates Datasets and DataLoaders."""
#     print("Loading data...")
#     try:
#         with open(train_file, "r", encoding='utf-8-sig') as f: lines_train = f.readlines()
#         with open(dev_file, "r", encoding='utf-8-sig') as f: lines_dev = f.readlines()
#         # Test file is loaded later in inference, but read for potential vocab usage if needed
#         try:
#             with open(test_file, "r", encoding='utf-8-sig') as f: lines_test = f.readlines()
#         except FileNotFoundError:
#             print(f"Warning: Test file '{test_file}' not found during initial load.")
#             lines_test = []
#     except FileNotFoundError as e:
#         print(f"Error loading data files: {e}"); raise

#     # Build vocabulary based on training data
#     tokenizer, tokenizer_inv, vocab_size = build_vocabulary(lines_train, min_freq=2)
#     pad_id = tokenizer[PAD_TOKEN]

#     # Create Datasets
#     train_dataset = ShakespeareWordDataset(lines_train, tokenizer, max_len)
#     val_dataset = ShakespeareWordDataset(lines_dev, tokenizer, max_len)
#     # test_dataset can be created similarly if needed for evaluation during training

#     # Create DataLoaders
#     train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
#     val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

#     return train_loader, val_loader, tokenizer, tokenizer_inv, vocab_size, pad_id

# # --- Helper Functions (Adapted for Word Level) ---

# def decode_tokens(tokens, tokenizer_inv, stop_at_stop=True, omit_pad=True, omit_start=True):
#     """Decodes a list/tensor of word IDs back to a string."""
#     words = []
#     # If it's a tensor, move to CPU and convert to list
#     if isinstance(tokens, torch.Tensor):
#         tokens = tokens.cpu().numpy().tolist()

#     for token_id in tokens:
#         word = tokenizer_inv.get(token_id, UNK_TOKEN) # Use UNK if ID not found
#         if stop_at_stop and word == STOP_TOKEN: break
#         if omit_pad and word == PAD_TOKEN: continue
#         if omit_start and word == START_TOKEN: continue
#         words.append(word)
#     # Join words, handling potential punctuation spacing issues slightly better
#     text = " ".join(words)
#     text = re.sub(r'\s([.,!?;:])', r'\1', text) # Attach punctuation to preceding word
#     return text

# # --- Transformer Model Components (Largely Reusable) ---
# # Use the same CausalSelfAttention, FeedForward, MultiHeadAttention, TransformerBlock
# # from the character-level implementation. Just ensure config passed matches.

# class CausalSelfAttention(nn.Module):
#     """ Single head of self-attention with causal masking. """
#     def __init__(self, config):
#         super().__init__()
#         assert config.n_embd % config.n_head == 0
#         self.head_dim = config.n_embd // config.n_head
#         # Key, query, value projections for all heads, but in a batch
#         self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
#         # Output projection
#         self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=False)
#         # Regularization
#         self.attn_dropout = nn.Dropout(config.dropout)
#         self.resid_dropout = nn.Dropout(config.dropout)
#         self.n_head = config.n_head
#         self.n_embd = config.n_embd
#         # Causal mask
#         # Create fixed causal mask buffer compatible with config.block_size
#         self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
#                              .view(1, 1, config.block_size, config.block_size))

#     def forward(self, x):
#         B, T, C = x.size() # Batch size, sequence length, embedding dimensionality (n_embd)

#         # Calculate query, key, values for all heads in batch and move head forward to be the batch dim
#         q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
#         q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2) # (B, nh, T, hs)
#         k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2) # (B, nh, T, hs)
#         v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2) # (B, nh, T, hs)

#         # Causal self-attention; Self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
#         att = (q @ k.transpose(-2, -1)) * (k.size(-1)**-0.5) # Scale
#         # Use the pre-computed mask up to sequence length T
#         att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf')) # Apply mask
#         att = F.softmax(att, dim=-1) # Softmax
#         att = self.attn_dropout(att)
#         # Weighted aggregation of values
#         y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
#         y = y.transpose(1, 2).contiguous().view(B, T, C) # Re-assemble all head outputs side by side

#         # Output projection
#         y = self.resid_dropout(self.c_proj(y))
#         return y

# class FeedForward(nn.Module):
#     """ Position-wise Feed-forward network. """
#     def __init__(self, config):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(config.n_embd, 4 * config.n_embd, bias=False),
#             nn.GELU(), # Changed from ReLU to GELU, common in newer Transformers
#             nn.Linear(4 * config.n_embd, config.n_embd, bias=False),
#             nn.Dropout(config.dropout),
#         )
#     def forward(self, x): return self.net(x)

# class MultiHeadAttention(nn.Module): # Wrapper - could just use CausalSelfAttention directly
#     def __init__(self, config): super().__init__(); self.attention = CausalSelfAttention(config)
#     def forward(self, x): return self.attention(x)

# class TransformerBlock(nn.Module):
#     """ Single Transformer block with Pre-LN structure. """
#     def __init__(self, config):
#         super().__init__()
#         self.ln_1 = nn.LayerNorm(config.n_embd)
#         self.attn = MultiHeadAttention(config) # Uses our CausalSelfAttention
#         self.ln_2 = nn.LayerNorm(config.n_embd)
#         self.ffn = FeedForward(config)
#     def forward(self, x):
#         x = x + self.attn(self.ln_1(x)) # Residual connection after attention
#         x = x + self.ffn(self.ln_2(x))  # Residual connection after FFN
#         return x

# class TransformerLM(nn.Module):
#     """ The full Transformer Language Model (Decoder-only). """
#     def __init__(self, config):
#         super().__init__()
#         self.config = config
#         self.pad_id = config.pad_id

#         # Embeddings + Positional Encoding
#         self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd, padding_idx=config.pad_id) # Use padding_idx
#         self.positional_embedding = nn.Embedding(config.block_size, config.n_embd) # block_size == MAX_LEN
#         self.dropout = nn.Dropout(config.dropout)

#         # Transformer Blocks
#         self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layer)])

#         # Final Layer Norm and LM Head
#         self.layer_norm_final = nn.LayerNorm(config.n_embd)
#         self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

#         # Weight tying (optional but good practice)
#         self.token_embedding.weight = self.lm_head.weight

#         # Init weights
#         self.apply(self._init_weights)
#         # Special init for residual projections
#         for pn, p in self.named_parameters():
#             if pn.endswith('c_proj.weight'):
#                 torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

#         print(f"TransformerLM (Word-Level) initialized.")
#         print(f" - Vocab Size: {config.vocab_size}")
#         print(f" - Embedding Dim: {config.n_embd}")
#         print(f" - Block Size (MAX_LEN): {config.block_size}")
#         print(f" - Layers: {config.n_layer}")
#         print(f" - Heads: {config.n_head}")
#         print(f" - Total Params: {sum(p.numel() for p in self.parameters())/1e6:.2f} M")
#         print(f" - Padding ID: {self.pad_id}")


#     def _init_weights(self, module):
#         if isinstance(module, nn.Linear):
#             torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
#             if module.bias is not None: torch.nn.init.zeros_(module.bias)
#         elif isinstance(module, nn.Embedding):
#             # Don't re-init padding idx if using weight tying and it's already set
#             if hasattr(module, 'padding_idx') and module.padding_idx is not None:
#                  with torch.no_grad():
#                      module.weight[module.padding_idx].fill_(0) # Ensure padding embedding is zero
#             torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
#         elif isinstance(module, nn.LayerNorm):
#             torch.nn.init.zeros_(module.bias)
#             torch.nn.init.ones_(module.weight)

#     def forward(self, idx, targets=None):
#         B, T = idx.size() # Batch size, Sequence length (T should == MAX_LEN / block_size)
#         assert T <= self.config.block_size, f"Input sequence length ({T}) exceeds block size ({self.config.block_size})"

#         # Get token embeddings
#         tok_emb = self.token_embedding(idx) # (B, T, n_embd)
#         # Get positional embeddings
#         pos = torch.arange(0, T, dtype=torch.long, device=idx.device).unsqueeze(0) # (1, T)
#         pos_emb = self.positional_embedding(pos) # (1, T, n_embd)

#         # Combine embeddings and apply dropout
#         x = self.dropout(tok_emb + pos_emb)

#         # Pass through Transformer blocks
#         for block in self.blocks:
#             x = block(x)

#         # Final layer norm
#         x = self.layer_norm_final(x) # (B, T, n_embd)

#         # Calculate logits
#         logits = self.lm_head(x) # (B, T, vocab_size)

#         # Calculate loss if targets are provided
#         loss = None
#         if targets is not None:
#             # Reshape for CrossEntropyLoss: (B*T, vocab_size), (B*T)
#             # Use ignore_index to automatically skip PAD tokens in the target
#             loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=self.pad_id)

#         return logits, loss

#     @torch.no_grad()
#     def generate(self, start_tokens, max_new_tokens, tokenizer, temperature=1.0, top_k=None):
#         """
#         Generate text sequences word by word.
#         start_tokens: Tensor of shape (1, N) with initial token IDs (including START_TOKEN).
#         """
#         self.eval()
#         idx = start_tokens.to(DEVICE)
#         stop_token_id = tokenizer[STOP_TOKEN]
#         pad_token_id = tokenizer[PAD_TOKEN]

#         for _ in range(max_new_tokens):
#             # Crop context if it exceeds block size
#             idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]

#             # Forward pass to get logits for the next token
#             logits, _ = self(idx_cond) # Logits shape (1, T, vocab_size)
#             # Pluck the logits for the final step
#             logits = logits[:, -1, :] / temperature # Shape (1, vocab_size)

#             # Optionally crop the logits to only the top k options
#             if top_k is not None:
#                 v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
#                 logits[logits < v[:, [-1]]] = -float('Inf')

#             # Apply softmax to get probabilities
#             probs = F.softmax(logits, dim=-1) # Shape (1, vocab_size)

#             # Sample the next token ID from the distribution
#             idx_next = torch.multinomial(probs, num_samples=1) # Shape (1, 1)

#             # Stop if we generate STOP token
#             if idx_next.item() == stop_token_id:
#                 break

#             # Append sampled token ID to the running sequence
#             idx = torch.cat((idx, idx_next), dim=1)

#             # Stop if sequence length exceeds max length (shouldn't happen if max_new_tokens is reasonable)
#             if idx.size(1) >= self.config.block_size:
#                  print("Warning: Generation reached max block size.")
#                  break


#         # Return the generated sequence (excluding the initial start token if desired)
#         return idx # Contains START token at the beginning

# # Learning rate decay scheduler (cosine with warmup) - Reuse from char level
# def get_lr(it):
#     # 1) linear warmup for warmup_iters steps
#     if it < WARMUP_ITERS: return LEARNING_RATE * it / WARMUP_ITERS
#     # 2) if it > lr_decay_iters, return min learning rate
#     if it > LR_DECAY_ITERS: return MIN_LR
#     # 3) in between, use cosine decay down to min learning rate
#     decay_ratio = (it - WARMUP_ITERS) / (LR_DECAY_ITERS - WARMUP_ITERS)
#     assert 0 <= decay_ratio <= 1
#     coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio)) # coeff starts at 1 and goes to 0
#     return MIN_LR + coeff * (LEARNING_RATE - MIN_LR)

# # --- Evaluation Function (Crucially handles padding) ---
# @torch.no_grad()
# def estimate_loss_and_perplexity(model, loader, device, pad_id):
#     """ Estimates average loss and perplexity over the given DataLoader. """
#     model.eval()
#     total_loss = 0.0
#     total_tokens = 0 # Count non-pad target tokens
#     num_batches = 0

#     for X, Y in loader: # Use tqdm(loader) for progress bar
#         X, Y = X.to(device), Y.to(device)
#         logits, loss = model(X, Y) # Loss is already calculated with ignore_index=pad_id

#         # Accumulate loss * number of non-pad tokens in the batch targets
#         # loss is the average loss *per non-pad token* in the batch
#         # We need total loss sum, so multiply by number of non-pad tokens
#         mask = (Y != pad_id)
#         num_non_pad = mask.sum().item()

#         if num_non_pad > 0:
#              total_loss += loss.item() * num_non_pad
#              total_tokens += num_non_pad
#         num_batches += 1
#         if num_batches >= EVAL_ITERS: # Limit evaluation iterations if needed
#             break

#     model.train() # Set back to train mode

#     if total_tokens == 0:
#         print("Warning: No non-pad tokens found during evaluation!")
#         return float('inf'), float('inf') # Avoid division by zero

#     average_loss = total_loss / total_tokens
#     perplexity = math.exp(average_loss) if average_loss < 700 else float('inf')

#     return average_loss, perplexity


# # --- Training Function ---
# def train_model(model, optimizer, train_loader, val_loader, config, tokenizer, tokenizer_inv):
#     """ Implements the training loop for word-level model. """
#     metrics = {'train_loss': [], 'val_loss': [], 'train_ppl': [], 'val_ppl': [], 'lr': [], 'step': []}
#     best_val_ppl = float('inf')
#     start_time = time.time()
#     iter_num = 0 # Track iterations instead of epochs if using MAX_ITERS

#     print(f"Starting training for ~{config.max_iters} iterations...")
#     model.train()

#     # Determine number of epochs needed to approx reach max_iters
#     # This is just for info, the loop runs based on iter_num
#     iters_per_epoch = len(train_loader)
#     num_epochs = (config.max_iters + iters_per_epoch - 1) // iters_per_epoch
#     print(f"Data loader has {iters_per_epoch} batches per epoch. Aiming for ~{num_epochs} epochs.")

#     # Training loop
#     epoch = 0
#     while iter_num < config.max_iters:
#         epoch_start_time = time.time()
#         epoch += 1
#         print(f"--- Starting Epoch {epoch} ---")
#         pbar = tqdm(train_loader, desc=f"Epoch {epoch} Training", leave=False)
#         for xb, yb in pbar:
#             # Update Learning Rate
#             lr = get_lr(iter_num)
#             for param_group in optimizer.param_groups: param_group['lr'] = lr

#             # Move batch to device
#             xb, yb = xb.to(DEVICE), yb.to(DEVICE)

#             # Forward pass & loss calculation (loss ignores padding)
#             logits, loss = model(xb, yb)

#             # Backward pass & optimization
#             optimizer.zero_grad(set_to_none=True)
#             loss.backward()
#             torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Gradient clipping
#             optimizer.step()

#             # Update progress bar
#             pbar.set_postfix({'loss': f"{loss.item():.4f}", 'lr': f"{lr:.6f}"})

#             # Log and Evaluate periodically
#             if iter_num % config.eval_interval == 0 or iter_num == config.max_iters - 1:
#                 time_elapsed = time.time() - start_time
#                 val_loss, val_ppl = estimate_loss_and_perplexity(model, val_loader, DEVICE, config.pad_id)

#                 # Estimate train loss/ppl on a subset for quick check (optional)
#                 # train_loss_est, train_ppl_est = estimate_loss_and_perplexity(model, train_loader, DEVICE, config.pad_id) # Can be slow

#                 # Use current batch loss as proxy for train loss for faster logging
#                 train_loss_proxy = loss.item()
#                 train_ppl_proxy = math.exp(train_loss_proxy) if train_loss_proxy < 700 else float('inf')

#                 print(f"\nIter {iter_num}: Train Loss (batch) {train_loss_proxy:.4f}, PPL {train_ppl_proxy:.2f} | "
#                       f"Val Loss {val_loss:.4f}, PPL {val_ppl:.2f} | LR {lr:.6f} | Time {time_elapsed:.1f}s")

#                 metrics['train_loss'].append(train_loss_proxy) # Log batch loss as proxy
#                 metrics['val_loss'].append(val_loss)
#                 metrics['train_ppl'].append(train_ppl_proxy) # Log proxy PPL
#                 metrics['val_ppl'].append(val_ppl)
#                 metrics['lr'].append(lr)
#                 metrics['step'].append(iter_num)

#                 if val_ppl < best_val_ppl:
#                     best_val_ppl = val_ppl
#                     print(f" >> New best val PPL: {best_val_ppl:.4f}. Saving model to {MODEL_SAVE_PATH}...")
#                     torch.save(model.state_dict(), MODEL_SAVE_PATH)

#                 # Generate sample text
#                 model.eval() # Ensure model is in eval mode for generation
#                 start_context_str = START_TOKEN
#                 start_tokens = torch.tensor([tokenize_line(start_context_str, tokenizer, config.block_size, add_start_stop=False)], dtype=torch.long) # Don't add stop here
#                 start_tokens = start_tokens[:, :1] # Just keep the START token ID
#                 generated_ids = model.generate(start_tokens, max_new_tokens=50, tokenizer=tokenizer, temperature=0.7)
#                 generated_text = decode_tokens(generated_ids[0], tokenizer_inv, stop_at_stop=True, omit_pad=True, omit_start=True)
#                 print(f"Sample Gen:\n---\n{generated_text}\n---")
#                 model.train() # Back to training mode


#             iter_num += 1
#             if iter_num >= config.max_iters: break # Exit inner loop if max_iters reached

#         epoch_time = time.time() - epoch_start_time
#         print(f"--- Epoch {epoch} Finished ({epoch_time:.2f}s) ---")

#     print("Training finished.")
#     print(f"Total Training Time: {time.time() - start_time:.2f} seconds")
#     print(f"Best Validation Perplexity: {best_val_ppl:.4f}")
#     return metrics

# # --- Text Generation Function ---
# def generate_sample(model, tokenizer, tokenizer_inv, context=START_TOKEN, gen_tokens=50, temperature=0.7):
#     """ Generate text using the model's generate method. """
#     print(f"\nGenerating {gen_tokens} tokens from context: '{context}'")
#     model.eval() # Ensure eval mode

#     # Tokenize context
#     context_tokens = tokenize_line(context, tokenizer, MAX_LEN, add_start_stop=False) # No stop/pad needed for start
#     # Keep only up to MAX_LEN, potentially less if context is short
#     context_tensor = torch.tensor([context_tokens], dtype=torch.long, device=DEVICE)
#     context_tensor = context_tensor[:, :MAX_LEN] # Ensure it fits model input size

#     with torch.no_grad():
#         generated_ids_full = model.generate(
#             context_tensor, max_new_tokens=gen_tokens, tokenizer=tokenizer, temperature=temperature
#         )[0] # Get the first (only) batch item

#     full_text = decode_tokens(generated_ids_full, tokenizer_inv, stop_at_stop=True, omit_pad=True, omit_start=False) # Keep START if generated
#     # Extract only the newly generated part
#     gen_part = full_text[len(decode_tokens(context_tensor[0], tokenizer_inv, omit_pad=True, omit_start=False)):]
#     gen_part = gen_part.strip()

#     return full_text, gen_part


# # --- Main Function ---
# def main():
#     # Load data and create loaders
#     train_loader, val_loader, tokenizer, tokenizer_inv, vocab_size, pad_id = \
#         load_and_preprocess_data(TRAIN_FILE, DEV_FILE, TEST_FILE_DEMO, MAX_LEN, BATCH_SIZE)

#     # Config Namespace
#     # Use MAX_LEN as the block_size for the model config
#     config = SimpleNamespace(
#         block_size=MAX_LEN, vocab_size=vocab_size, n_layer=N_LAYER, n_head=N_HEAD,
#         n_embd=N_EMBD, dropout=DROPOUT, max_iters=MAX_ITERS, eval_interval=EVAL_INTERVAL,
#         learning_rate=LEARNING_RATE, eval_iters=EVAL_ITERS, pad_id=pad_id # Pass pad_id
#     )

#     # Init Model & Optimizer
#     model = TransformerLM(config)
#     model.to(DEVICE)
#     optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=WEIGHT_DECAY)

#     # Train
#     metrics = train_model(model, optimizer, train_loader, val_loader, config, tokenizer, tokenizer_inv)

#     # Plot Metrics
#     if metrics and metrics['step']: # Check if metrics were generated
#         fig, ax1 = plt.subplots(figsize=(12, 6))
#         color = 'tab:red'
#         ax1.set_xlabel('Iterations')
#         ax1.set_ylabel('Loss', color=color)
#         # Plot smoothed loss if desired, or direct values
#         # Use val loss directly, train loss is batch proxy so might be noisy
#         ax1.plot(metrics['step'], metrics['train_loss'], color='lightcoral', linestyle='--', label='Train Loss (Batch Proxy)')
#         ax1.plot(metrics['step'], metrics['val_loss'], color=color, label='Val Loss')
#         ax1.tick_params(axis='y', labelcolor=color)
#         ax1.legend(loc='upper left')
#         ax1.grid(True, axis='y')
#         # ax1.set_ylim(bottom=0) # Adjust ylim as needed

#         ax2 = ax1.twinx()
#         color = 'tab:blue'
#         ax2.set_ylabel('Learning Rate', color=color)
#         ax2.plot(metrics['step'], metrics['lr'], color=color, label='LR')
#         ax2.tick_params(axis='y', labelcolor=color)
#         ax2.legend(loc='upper right')

#         fig.tight_layout()
#         plt.title('Word-Level Transformer: Loss & Learning Rate')
#         plt.savefig(PLOT_SAVE_PATH)
#         print(f"\nLoss/LR plot saved: {PLOT_SAVE_PATH}")
#         plt.close(fig) # Close figure

#         # Perplexity Plot
#         fig_ppl, ax_ppl = plt.subplots(figsize=(10, 5))
#         ax_ppl.plot(metrics['step'], metrics['train_ppl'], label='Train PPL (Batch Proxy)', linestyle='--', color='lightblue')
#         ax_ppl.plot(metrics['step'], metrics['val_ppl'], label='Val PPL', color='blue')
#         ax_ppl.set_xlabel('Iterations')
#         ax_ppl.set_ylabel('Perplexity')
#         ax_ppl.set_title('Word-Level Transformer: Perplexity')
#         ax_ppl.legend()
#         ax_ppl.grid(True)
#         ax_ppl.set_ylim(bottom=0, top=min(max(metrics['val_ppl'])*1.2, 300)) # Cap y-axis for readability
#         ppl_plot_path = PLOT_SAVE_PATH.replace('.png', '_perplexity.png')
#         plt.savefig(ppl_plot_path)
#         print(f"Perplexity plot saved: {ppl_plot_path}")
#         plt.close(fig_ppl)

#     # Final Evaluation on Test Set (using the inference function)
#     print(f"\n--- Running Inference on {TEST_FILE_DEMO} using best model {MODEL_SAVE_PATH} ---")
#     # Create dummy test file if needed for the flow
#     if not os.path.exists(TEST_FILE_DEMO):
#         print(f"Warning: Test file '{TEST_FILE_DEMO}' not found. Creating dummy.")
#         with open(TEST_FILE_DEMO, "w", encoding='utf-8') as f:
#              f.write("First Citizen:\n")
#              f.write("To be, or not to be, that is the question:\n")


#     generated_texts, test_ppl = inference(
#         model_path=MODEL_SAVE_PATH,
#         test_file=TEST_FILE_DEMO,
#         tokenizer=tokenizer,
#         tokenizer_inv=tokenizer_inv,
#         config=config, # Pass the model config
#         gen_tokens=50,
#         temperature=0.7
#     )

#     print(f"\n--- Final Evaluation (Best Model: {MODEL_SAVE_PATH}) ---")
#     if test_ppl is not None:
#         print(f"Test Perplexity on '{TEST_FILE_DEMO}': {test_ppl:.4f}")
#     else:
#         print(f"Test Perplexity: N/A (Could not calculate)")

#     print("\nSample Generations from Test File:")
#     if generated_texts:
#         for i, item in enumerate(generated_texts[:5]): # Show first 5 examples
#             print(f"[{i+1}] Context: {item['context']}")
#             print(f"    Generated: {item['generated']}")
#             print("-" * 15)
#     else:
#         print("No text generated or error during inference.")


# # --- Inference Function ---
# def inference(model_path, test_file, tokenizer, tokenizer_inv, config, gen_tokens=50, temperature=0.6):
#     """ Loads model, runs generation and PPL calculation on test file. """
#     print("\n--- Starting Inference ---")
#     generated_texts = []
#     test_perplexity = None

#     # Load Model
#     try:
#         print(f"Loading model from {model_path}...")
#         # Recreate model using saved config implicitly via passed 'config' object
#         # Ensure config has vocab_size and pad_id matching the loaded model's training
#         model = TransformerLM(config)
#         model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True)) # Use weights_only=True if using torch 1.13+
#         model.to(DEVICE)
#         model.eval()
#         print("Model loaded.")
#     except FileNotFoundError:
#         print(f"Error: Model file '{model_path}' not found.")
#         return [], None
#     except Exception as e:
#         print(f"Error loading model: {e}")
#         return [], None

#     # --- Calculate Perplexity on Test Set ---
#     try:
#         print(f"\nReading test file for PPL: {test_file}")
#         with open(test_file, "r", encoding='utf-8-sig') as f:
#             test_lines = f.readlines()

#         if not test_lines:
#             print("Warning: Test file is empty. Cannot calculate perplexity.")
#         else:
#             # Create a temporary Dataset and DataLoader for the test set
#             test_dataset = ShakespeareWordDataset(test_lines, tokenizer, config.block_size) # Use model's block size
#             if len(test_dataset) == 0:
#                  print("Warning: No valid sequences generated from test file. Cannot calculate perplexity.")
#             else:
#                 test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False) # Use same batch size as eval?
#                 print(f"Calculating PPL on {len(test_dataset)} test sequences...")
#                 _, test_perplexity = estimate_loss_and_perplexity(model, test_loader, DEVICE, config.pad_id)
#                 print(f"Test Perplexity: {test_perplexity:.4f}")

#     except FileNotFoundError:
#         print(f"Error: Test file '{test_file}' not found for PPL calculation.")
#     except Exception as e:
#         print(f"Error during PPL calculation: {e}")
#         test_perplexity = None # Ensure PPL is None if error occurs

#     # --- Generate Text for each line in Test Set ---
#     try:
#         print(f"\nGenerating text for contexts from {test_file}...")
#         with open(test_file, "r", encoding='utf-8-sig') as f:
#              test_lines = f.readlines() # Re-read lines

#         for line in test_lines[:10]: # Limit examples for output
#             context = line.strip()
#             if not context: continue
#             print(f"\nContext: {context}")

#             # Check for unknown words in context - generation might be poor
#             context_words = simple_word_tokenize(context)
#             unknowns = [w for w in context_words if w not in tokenizer]
#             if unknowns: print(f"  (Context contains unknown words: {unknowns[:5]}{'...' if len(unknowns)>5 else ''})")

#             # Use the generation helper
#             full_text, generated_part = generate_sample(model, tokenizer, tokenizer_inv, context=context, gen_tokens=gen_tokens, temperature=temperature)

#             print(f"Generated: {generated_part}")
#             generated_texts.append({"context": context, "generated": generated_part})

#     except FileNotFoundError:
#         print(f"Error: Test file '{test_file}' not found for generation.")
#     except Exception as e:
#         print(f"Error during text generation: {e}")

#     return generated_texts, test_perplexity


# # SimpleNamespace shim for non-notebook environments
# try:
#     from argparse import Namespace as SimpleNamespace
# except ImportError:
#     # Define a simple class if argparse is not available
#     class SimpleNamespace:
#         def __init__(self, **kwargs):
#             self.__dict__.update(kwargs)

# # --- Main Execution Guard ---
# if __name__ == "__main__":
#     main()

In [3]:
# task1_word_level_improved.py

import torch
import torch.nn as nn
from torch.nn import functional as F
import math
import time
import os
from collections import Counter
from tqdm import tqdm
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import numpy as np
import re # For basic word splitting

# --- Configuration ---
# Kaggle Environment Check
IS_KAGGLE = os.path.exists('/kaggle/input')
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Hyperparameters (Adjusted for Better Performance)
BATCH_SIZE = 32        # Adjusted batch size (check GPU memory)
MAX_LEN = 128          # Increased Max sequence length (words)
MAX_ITERS = 25000      # Increased training iterations significantly
EVAL_INTERVAL = 250    # Evaluate less often due to longer training
LEARNING_RATE = 3e-4   # Standard learning rate for medium Transformers
EVAL_ITERS = 100       # Batches used for loss estimation
N_EMBD = 384           # Increased Embedding dimension
N_HEAD = 6             # Increased Number of attention heads
N_LAYER = 6            # Increased Number of transformer blocks
DROPOUT = 0.2          # Slightly increased Regularization
WEIGHT_DECAY = 0.1     # Standard Regularization

# --- Learning Rate Schedule Parameters ---
WARMUP_ITERS = 200     # Increase warmup slightly
LR_DECAY_ITERS = MAX_ITERS # Decay over the full training duration
MIN_LR = 3e-5          # Minimum learning rate (1/10th of max)

# Data Paths
BASE_DIR = '/kaggle/input/nlp-a3-dataset/' if IS_KAGGLE else './'
TRAIN_FILE = os.path.join(BASE_DIR, 'shakespear_train.txt')
DEV_FILE = os.path.join(BASE_DIR, 'shakespear_dev.txt')
TEST_FILE_DEMO = 'shakespear_test.txt' # Expected name for demo test file

# Output Paths
MODEL_SAVE_PATH = 'task1_transformer_word_level_improved.pth'
PLOT_SAVE_PATH = 'task1_loss_lr_plot_word_level_improved.png'

# Ensure N_EMBD is divisible by N_HEAD
assert N_EMBD % N_HEAD == 0

# Seed for reproducibility
torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed(1337)

# --- Special Tokens ---
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
START_TOKEN = "<START>"
STOP_TOKEN = "<STOP>"
special_tokens = [PAD_TOKEN, UNK_TOKEN, START_TOKEN, STOP_TOKEN]

# --- Word Tokenization and Preprocessing (Same as before) ---

def simple_word_tokenize(text):
    """Basic word tokenizer: lowercase, split by space/punctuation."""
    text = text.lower()
    words = re.findall(r"[\w']+|[.,!?;:]", text)
    return words

def build_vocabulary(all_lines, min_freq=2):
    """Builds word vocabulary from tokenized lines."""
    print("Building word vocabulary...")
    token_counts = Counter()
    # Use tqdm for progress if list is long
    lines_iterator = tqdm(all_lines, desc="Counting words") if len(all_lines) > 1000 else all_lines
    for line in lines_iterator:
        token_counts.update(simple_word_tokenize(line))

    vocab = special_tokens[:] # Start with special tokens
    for token, count in token_counts.items():
        if count >= min_freq:
            vocab.append(token)
    print(f"Vocabulary size: {len(vocab)} (min_freq={min_freq})")
    print(f"Sample vocab: {vocab[:10]} ... {vocab[-10:]}")

    tokenizer = {token: i for i, token in enumerate(vocab)}
    tokenizer_inv = {i: token for token, i in tokenizer.items()}
    for st in special_tokens: # Sanity check
        if st not in tokenizer: raise ValueError(f"Special token '{st}' missing!")
    return tokenizer, tokenizer_inv, len(vocab)

def tokenize_line(line, tokenizer, max_len, add_start_stop=True):
    """Tokenizes a single line, adds special tokens, handles UNK, and pads/truncates."""
    words = simple_word_tokenize(line)
    tokens = []
    if add_start_stop: tokens.append(tokenizer[START_TOKEN])
    for word in words: tokens.append(tokenizer.get(word, tokenizer[UNK_TOKEN]))
    if add_start_stop: tokens.append(tokenizer[STOP_TOKEN])
    tokens = tokens[:max_len] # Truncate
    padding_needed = max_len - len(tokens)
    if padding_needed > 0: tokens.extend([tokenizer[PAD_TOKEN]] * padding_needed) # Pad
    return tokens

class ShakespeareWordDataset(Dataset):
    def __init__(self, lines, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.max_len = max_len # Max length INCLUDING start/stop tokens
        self.pad_id = tokenizer[PAD_TOKEN]
        print(f"Tokenizing {len(lines)} lines for dataset (max_len={max_len})...")
        self.data = []
        lines_iterator = tqdm(lines, desc="Tokenizing lines") if len(lines) > 1000 else lines
        for line in lines_iterator:
            line_strip = line.strip()
            if not line_strip: continue
            # Tokenize to max_len + 1 for creating X, Y pairs of length max_len
            token_ids = tokenize_line(line_strip, tokenizer, max_len + 1, add_start_stop=True)
            # Ensure we have at least START + WORD + STOP/PAD (len > 2)
            if len(token_ids) > 2 and token_ids[0] == tokenizer[START_TOKEN]:
                # Check if sequence contains non-pad tokens besides START/STOP
                non_pad_count = sum(1 for tid in token_ids if tid != self.pad_id)
                if non_pad_count > 2: # Needs START + word + STOP (or another word)
                    self.data.append(torch.tensor(token_ids, dtype=torch.long))

        print(f"Created dataset with {len(self.data)} valid sequences.")
        if not self.data: print("Warning: Dataset is empty after filtering!")

    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        full_seq = self.data[idx]
        x = full_seq[:-1] # Input: <START> w1 ... wn <STOP/PAD> (len=max_len)
        y = full_seq[1:]  # Target: w1 ... wn <STOP/PAD> <PAD> (len=max_len)
        assert x.shape[0] == self.max_len and y.shape[0] == self.max_len, "Shape mismatch!"
        return x, y

def load_and_preprocess_data(train_file, dev_file, test_file, max_len, batch_size):
    """Loads data, builds vocab, creates Datasets and DataLoaders."""
    print("Loading data files...")
    try:
        with open(train_file, "r", encoding='utf-8-sig') as f: lines_train = f.readlines()
        with open(dev_file, "r", encoding='utf-8-sig') as f: lines_dev = f.readlines()
        try:
            with open(test_file, "r", encoding='utf-8-sig') as f: lines_test = f.readlines()
        except FileNotFoundError: print(f"Info: Test file '{test_file}' not found during initial load."); lines_test = []
    except FileNotFoundError as e: print(f"Error loading data files: {e}"); raise

    tokenizer, tokenizer_inv, vocab_size = build_vocabulary(lines_train, min_freq=2)
    pad_id = tokenizer[PAD_TOKEN]

    train_dataset = ShakespeareWordDataset(lines_train, tokenizer, max_len)
    val_dataset = ShakespeareWordDataset(lines_dev, tokenizer, max_len)

    if len(train_dataset) == 0 or len(val_dataset) == 0:
        raise ValueError("Training or Validation dataset is empty. Check data processing/filtering.")

    # Use num_workers > 0 if not debugging DataLoader issues
    num_workers = 2 if DEVICE == 'cuda' else 0 # Can cause issues on some setups
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True if DEVICE == 'cuda' else False)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True if DEVICE == 'cuda' else False)

    return train_loader, val_loader, tokenizer, tokenizer_inv, vocab_size, pad_id

# --- Helper Functions (Adapted for Word Level) ---
def decode_tokens(tokens, tokenizer_inv, stop_at_stop=True, omit_pad=True, omit_start=True):
    """Decodes a list/tensor of word IDs back to a string."""
    words = []
    if isinstance(tokens, torch.Tensor): tokens = tokens.cpu().numpy().tolist()
    for token_id in tokens:
        word = tokenizer_inv.get(token_id, UNK_TOKEN)
        if stop_at_stop and word == STOP_TOKEN: break
        if omit_pad and word == PAD_TOKEN: continue
        if omit_start and word == START_TOKEN: continue
        words.append(word)
    text = " ".join(words)
    text = re.sub(r'\s([.,!?;:])', r'\1', text) # Basic punctuation handling
    return text

# --- Transformer Model Components (Same structure, different config) ---
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.head_dim = config.n_embd // config.n_head
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=False)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head; self.n_embd = config.n_embd
        # Use register_buffer for non-parameter tensors that should be part of the state_dict
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                             .view(1, 1, config.block_size, config.block_size))
    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (k.size(-1)**-0.5)
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1); att = self.attn_dropout(att)
        y = att @ v; y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd, bias=False),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd, bias=False),
            nn.Dropout(config.dropout),
        )
    def forward(self, x): return self.net(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, config): super().__init__(); self.attention = CausalSelfAttention(config)
    def forward(self, x): return self.attention(x)

class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd); self.attn = MultiHeadAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd); self.ffn = FeedForward(config)
    def forward(self, x): x = x + self.attn(self.ln_1(x)); x = x + self.ffn(self.ln_2(x)); return x

class TransformerLM(nn.Module):
    def __init__(self, config):
        super().__init__(); self.config = config; self.pad_id = config.pad_id
        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd, padding_idx=config.pad_id)
        self.positional_embedding = nn.Embedding(config.block_size, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layer)])
        self.layer_norm_final = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.token_embedding.weight = self.lm_head.weight # Weight tying
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'): torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

        print(f"TransformerLM (Word-Level Improved) initialized.")
        print(f" - Vocab Size: {config.vocab_size}, Pad ID: {self.pad_id}")
        print(f" - Embedding Dim: {config.n_embd}, Block Size (MAX_LEN): {config.block_size}")
        print(f" - Layers: {config.n_layer}, Heads: {config.n_head}")
        print(f" - Dropout: {config.dropout}, Weight Decay: {WEIGHT_DECAY}") # Added WD
        print(f" - Total Params: {sum(p.numel() for p in self.parameters())/1e6:.2f} M")

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None: torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.padding_idx is not None:
                 with torch.no_grad(): module.weight[module.padding_idx].fill_(0) # Ensure pad emb is zero
        elif isinstance(module, nn.LayerNorm):
            torch.nn.init.zeros_(module.bias); torch.nn.init.ones_(module.weight)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.config.block_size, f"Seq len {T} > block size {self.config.block_size}"
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device).unsqueeze(0)
        tok_emb = self.token_embedding(idx); pos_emb = self.positional_embedding(pos)
        x = self.dropout(tok_emb + pos_emb)
        for block in self.blocks: x = block(x)
        x = self.layer_norm_final(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=self.pad_id)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, tokenizer, temperature=1.0, top_k=None):
        """
        Generate text sequences word by word. Handles context window internally.
        idx: Tensor of shape (B, T_ctx) with UNPADDED starting token IDs. B is usually 1.
        """
        self.eval()
        stop_token_id = tokenizer[STOP_TOKEN]

        for _ in range(max_new_tokens):
            # Crop context if it exceeds block size, ensuring T <= block_size for the forward pass
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]

            # Forward pass to get logits for the next token using the potentially cropped context
            logits, _ = self(idx_cond) # Logits shape (B, T_cond, vocab_size)
            logits = logits[:, -1, :] / temperature # Pluck the logits for the final step, apply T

            if top_k is not None: # Optional Top-K sampling
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')

            probs = F.softmax(logits, dim=-1) # Apply softmax
            idx_next = torch.multinomial(probs, num_samples=1) # Sample next token

            if idx_next.item() == stop_token_id: break # Stop if STOP token generated

            idx = torch.cat((idx, idx_next), dim=1) # Append sampled token ID

            # Note: We don't need an explicit length check here anymore,
            # because idx_cond handles the context window size passed to self().

        return idx # Return the full generated sequence (including original context)

# Learning rate decay scheduler (cosine with warmup) - Reuse from char level
def get_lr(it):
    if it < WARMUP_ITERS: return LEARNING_RATE * it / WARMUP_ITERS
    if it > LR_DECAY_ITERS: return MIN_LR
    decay_ratio = (it - WARMUP_ITERS) / (LR_DECAY_ITERS - WARMUP_ITERS)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return MIN_LR + coeff * (LEARNING_RATE - MIN_LR)

# --- Evaluation Function (Same as before, handles padding) ---
@torch.no_grad()
def estimate_loss_and_perplexity(model, loader, device, pad_id, max_eval_batches=EVAL_ITERS):
    """ Estimates average loss and perplexity over the given DataLoader. """
    model.eval()
    total_loss = 0.0
    total_tokens = 0 # Count non-pad target tokens
    num_batches = 0
    # Use tqdm only if evaluating many batches
    loader_iter = tqdm(loader, desc="Evaluating", leave=False) if max_eval_batches > 20 else loader

    for X, Y in loader_iter:
        if num_batches >= max_eval_batches: break
        X, Y = X.to(device), Y.to(device)
        logits, loss = model(X, Y) # Loss is already calculated with ignore_index=pad_id
        mask = (Y != pad_id)
        num_non_pad = mask.sum().item()
        if num_non_pad > 0:
             total_loss += loss.item() * num_non_pad # Accumulate total loss sum
             total_tokens += num_non_pad
        num_batches += 1

    model.train() # Set back to train mode
    if total_tokens == 0: return float('inf'), float('inf') # Avoid division by zero
    average_loss = total_loss / total_tokens
    perplexity = math.exp(average_loss) if average_loss < 700 else float('inf') # Cap exp for stability
    return average_loss, perplexity

# --- Training Function (Adjusted for longer training) ---
def train_model(model, optimizer, train_loader, val_loader, config, tokenizer, tokenizer_inv):
    """ Implements the training loop for word-level model. """
    metrics = {'train_loss': [], 'val_loss': [], 'train_ppl': [], 'val_ppl': [], 'lr': [], 'step': []}
    best_val_ppl = float('inf')
    start_time = time.time()
    iter_num = 0 # Global iteration counter

    print(f"Starting training for {config.max_iters} iterations...")
    model.train()
    epoch = 0
    # Training loop - breaks when iter_num reaches max_iters
    while iter_num < config.max_iters:
        epoch += 1
        print(f"\n--- Starting Epoch {epoch} ---")
        epoch_start_time = time.time()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch} Training", leave=True) # Leave=True might be better for long runs

        for xb, yb in pbar:
            if iter_num >= config.max_iters: break # Check if max iters reached mid-epoch

            lr = get_lr(iter_num)
            for param_group in optimizer.param_groups: param_group['lr'] = lr
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits, loss = model(xb, yb) # Forward pass, loss calculation handles padding
            optimizer.zero_grad(set_to_none=True) # Backward pass & optimization
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Gradient clipping
            optimizer.step()

            pbar.set_postfix({'loss': f"{loss.item():.4f}", 'PPL': f"{math.exp(loss.item()):.1f}", 'lr': f"{lr:.6f}"})

            # Log and Evaluate periodically
            if iter_num % config.eval_interval == 0 or iter_num == config.max_iters - 1:
                time_elapsed = time.time() - start_time
                val_loss, val_ppl = estimate_loss_and_perplexity(model, val_loader, DEVICE, config.pad_id)
                train_loss_proxy = loss.item() # Use current batch loss as proxy
                train_ppl_proxy = math.exp(train_loss_proxy) if train_loss_proxy < 700 else float('inf')

                print(f"\nIter {iter_num}: Train Loss (batch) {train_loss_proxy:.4f}, PPL {train_ppl_proxy:.2f} | "
                      f"Val Loss {val_loss:.4f}, PPL {val_ppl:.2f} | LR {lr:.6f} | Time {time_elapsed:.1f}s")

                metrics['train_loss'].append(train_loss_proxy)
                metrics['val_loss'].append(val_loss); metrics['train_ppl'].append(train_ppl_proxy)
                metrics['val_ppl'].append(val_ppl); metrics['lr'].append(lr); metrics['step'].append(iter_num)

                if val_ppl < best_val_ppl:
                    best_val_ppl = val_ppl
                    print(f" >> New best val PPL: {best_val_ppl:.4f}. Saving model to {MODEL_SAVE_PATH}...")
                    torch.save(model.state_dict(), MODEL_SAVE_PATH)
                else:
                    print(f" (Best Val PPL remains {best_val_ppl:.4f})")


                # Generate sample text (using the fixed function)
                model.eval() # Ensure eval mode
                sample_start_context = START_TOKEN
                _, generated_sample_text = generate_sample(model, tokenizer, tokenizer_inv, config, context=sample_start_context, gen_tokens=60, temperature=0.7)
                print(f"Sample Gen:\n---\n{generated_sample_text}\n---")
                model.train() # Back to training mode

            iter_num += 1 # Increment iteration counter

        epoch_time = time.time() - epoch_start_time
        pbar.close() # Close the tqdm bar for the epoch
        print(f"--- Epoch {epoch} Finished ({epoch_time:.2f}s). Total Iters: {iter_num} ---")


    print("\nTraining finished.")
    print(f"Total Training Time: {time.time() - start_time:.2f} seconds")
    print(f"Best Validation Perplexity achieved: {best_val_ppl:.4f}")
    return metrics

# --- Text Generation Function (FIXED) ---
def generate_sample(model, tokenizer, tokenizer_inv, config, context=START_TOKEN, gen_tokens=50, temperature=0.7):
    """ Generate text using the model's generate method. Handles context correctly. """
    # print(f"\nGenerating {gen_tokens} tokens from context: '{context}'") # Optional print
    model.eval() # Ensure eval mode

    # 1. Tokenize context string
    context_words = simple_word_tokenize(context)
    context_tokens = [tokenizer.get(w, tokenizer[UNK_TOKEN]) for w in context_words]

    # 2. Truncate context if longer than block_size - 1 (to allow space for generation)
    #    The model.generate method internally handles the sliding window view.
    context_tokens = context_tokens[-(config.block_size - 1):]

    # 3. Create UNPADDED tensor
    context_tensor = torch.tensor([context_tokens], dtype=torch.long, device=DEVICE)
    # print(f"  (Input context tensor shape to generate: {context_tensor.shape})") # Debug print

    with torch.no_grad():
        # 4. Call model.generate with the unpadded, potentially truncated context
        generated_ids_full = model.generate(
            context_tensor, max_new_tokens=gen_tokens, tokenizer=tokenizer, temperature=temperature
        )[0] # Get the first (only) batch item

    # 5. Decode the full sequence
    full_text = decode_tokens(generated_ids_full, tokenizer_inv, stop_at_stop=True, omit_pad=True, omit_start=False)

    # 6. Extract only the newly generated part robustly
    # Decode the original *input* tensor to find the prefix
    original_context_decoded = decode_tokens(context_tensor[0], tokenizer_inv, stop_at_stop=False, omit_pad=True, omit_start=False)
    original_context_decoded = original_context_decoded.strip() # Ensure no leading/trailing spaces interfere

    # Handle potential empty context after tokenization/UNK replacement
    if not original_context_decoded:
        gen_part = full_text # If input context was effectively empty, return everything
    elif full_text.startswith(original_context_decoded):
         gen_part = full_text[len(original_context_decoded):].strip()
    else:
        # Fallback if prefix matching fails (e.g., due to decode nuances)
        print(f"Warning: Prefix mismatch during generation extraction. Context:'{original_context_decoded}', Full:'{full_text}'")
        # Try to return something reasonable - maybe split and take the suffix?
        # For simplicity, return the full text minus the first token (usually START)
        gen_part = decode_tokens(generated_ids_full[1:], tokenizer_inv, stop_at_stop=True, omit_pad=True, omit_start=False)


    return full_text, gen_part.strip()


# --- Main Function (Adjusted Paths/Config) ---
def main():
    train_loader, val_loader, tokenizer, tokenizer_inv, vocab_size, pad_id = \
        load_and_preprocess_data(TRAIN_FILE, DEV_FILE, TEST_FILE_DEMO, MAX_LEN, BATCH_SIZE)

    config = SimpleNamespace(
        block_size=MAX_LEN, vocab_size=vocab_size, n_layer=N_LAYER, n_head=N_HEAD,
        n_embd=N_EMBD, dropout=DROPOUT, max_iters=MAX_ITERS, eval_interval=EVAL_INTERVAL,
        learning_rate=LEARNING_RATE, eval_iters=EVAL_ITERS, pad_id=pad_id
    )

    model = TransformerLM(config); model.to(DEVICE)
    # Consider gradient accumulation if batch size needs to be very small for memory
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95)) # Standard betas

    metrics = train_model(model, optimizer, train_loader, val_loader, config, tokenizer, tokenizer_inv)

    # Plot Metrics (Same plotting code)
    if metrics and metrics['step']:
        fig, ax1 = plt.subplots(figsize=(12, 6)); color = 'tab:red'
        ax1.set_xlabel('Iterations'); ax1.set_ylabel('Loss', color=color)
        ax1.plot(metrics['step'], metrics['train_loss'], color='lightcoral', linestyle='--', alpha=0.7, label='Train Loss (Batch Proxy)')
        ax1.plot(metrics['step'], metrics['val_loss'], color=color, label='Val Loss')
        ax1.tick_params(axis='y', labelcolor=color); ax1.legend(loc='upper left'); ax1.grid(True, axis='y')
        # Optional: Set ylim based on observed values
        min_val_loss = min(metrics['val_loss']) if metrics['val_loss'] else 0
        ax1.set_ylim(bottom=max(0, min_val_loss - 1.0), top=min_val_loss + 5.0) # Zoom slightly around min val loss

        ax2 = ax1.twinx(); color = 'tab:blue'; ax2.set_ylabel('Learning Rate', color=color)
        ax2.plot(metrics['step'], metrics['lr'], color=color, label='LR')
        ax2.tick_params(axis='y', labelcolor=color); ax2.legend(loc='upper right')
        fig.tight_layout(); plt.title('Word-Level Transformer (Improved): Loss & LR'); plt.savefig(PLOT_SAVE_PATH)
        print(f"\nLoss/LR plot saved: {PLOT_SAVE_PATH}"); plt.close(fig)

        fig_ppl, ax_ppl = plt.subplots(figsize=(10, 5))
        ax_ppl.plot(metrics['step'], metrics['train_ppl'], label='Train PPL (Batch Proxy)', linestyle='--', color='lightblue', alpha=0.7)
        ax_ppl.plot(metrics['step'], metrics['val_ppl'], label='Val PPL', color='blue')
        ax_ppl.set_xlabel('Iterations'); ax_ppl.set_ylabel('Perplexity'); ax_ppl.set_title('Word-Level Transformer (Improved): Perplexity')
        ax_ppl.legend(); ax_ppl.grid(True);
        min_val_ppl = min(metrics['val_ppl']) if metrics['val_ppl'] else 1
        max_val_ppl = max(metrics['val_ppl']) if metrics['val_ppl'] else 1000
        ax_ppl.set_ylim(bottom=0, top=min(min_val_ppl*2, max_val_ppl*1.1, 300)) # Adjust Y axis cap dynamically
        ppl_plot_path = PLOT_SAVE_PATH.replace('.png', '_perplexity.png'); plt.savefig(ppl_plot_path)
        print(f"Perplexity plot saved: {ppl_plot_path}"); plt.close(fig_ppl)

    # Final Evaluation on Test Set
    print(f"\n--- Running Inference on {TEST_FILE_DEMO} using best model {MODEL_SAVE_PATH} ---")
    if not os.path.exists(TEST_FILE_DEMO):
        print(f"Warning: Test file '{TEST_FILE_DEMO}' not found. Creating dummy.")
        with open(TEST_FILE_DEMO, "w", encoding='utf-8') as f:
             f.write("First Citizen:\n")
             f.write("To be, or not to be, that is the question:\n")

    # Run inference using the best saved model
    generated_texts, test_ppl = inference(
        model_path=MODEL_SAVE_PATH, test_file=TEST_FILE_DEMO, tokenizer=tokenizer,
        tokenizer_inv=tokenizer_inv, config=config, gen_tokens=60, temperature=0.7
    )

    print(f"\n--- Final Evaluation (Best Model: {MODEL_SAVE_PATH}) ---")
    if test_ppl is not None: print(f"Test Perplexity on '{TEST_FILE_DEMO}': {test_ppl:.4f}")
    else: print(f"Test Perplexity: N/A (Could not calculate)")

    print("\nSample Generations from Test File:")
    if generated_texts:
        for i, item in enumerate(generated_texts[:5]): # Show first 5 examples
            print(f"[{i+1}] Context: {item['context']}")
            print(f"    Generated: {item['generated']}")
            print("-" * 15)
    else: print("No text generated or error during inference.")

# --- Inference Function (Loads model, calls generate_sample, calculates PPL) ---
def inference(model_path, test_file, tokenizer, tokenizer_inv, config, gen_tokens=60, temperature=0.6):
    """ Loads model, runs generation and PPL calculation on test file. """
    print("\n--- Starting Inference ---"); generated_texts = []; test_perplexity = None
    try: # Load Model
        print(f"Loading model from {model_path}..."); model = TransformerLM(config)
        try: # Try loading with weights_only first for security/speed if available
             model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True))
        except TypeError: # Fallback for older PyTorch versions
             print("   (weights_only=True failed, trying normal load)")
             model.load_state_dict(torch.load(model_path, map_location=DEVICE))
        model.to(DEVICE); model.eval(); print("Model loaded.")
    except FileNotFoundError: print(f"Error: Model file '{model_path}' not found."); return [], None
    except Exception as e: print(f"Error loading model: {e}"); return [], None

    try: # Calculate Perplexity
        print(f"\nReading test file for PPL: {test_file}")
        with open(test_file, "r", encoding='utf-8-sig') as f: test_lines = f.readlines()
        if not test_lines: print("Warning: Test file empty, skipping PPL.")
        else:
            # Create temporary Dataset/DataLoader for test set
            test_dataset = ShakespeareWordDataset(test_lines, tokenizer, config.block_size)
            if len(test_dataset) == 0: print("Warning: No valid test sequences, skipping PPL.")
            else:
                test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False) # Use same batch size?
                print(f"Calculating PPL on {len(test_dataset)} test sequences...")
                _, test_perplexity = estimate_loss_and_perplexity(model, test_loader, DEVICE, config.pad_id, max_eval_batches=len(test_loader)) # Eval all test batches
                print(f"Test Perplexity: {test_perplexity:.4f}")
    except FileNotFoundError: print(f"Error: Test file '{test_file}' not found for PPL.")
    except Exception as e: print(f"Error during PPL calculation: {e}"); test_perplexity = None

    try: # Generate Text
        print(f"\nGenerating text for contexts from {test_file}...")
        with open(test_file, "r", encoding='utf-8-sig') as f: test_lines = f.readlines()
        for line in test_lines[:10]: # Limit examples
            context = line.strip();
            if not context: continue; print(f"\nContext: {context}")
            context_words = simple_word_tokenize(context)
            unknowns = [w for w in context_words if w not in tokenizer]
            if unknowns: print(f"  (Context has unknown words: {unknowns[:3]}{'...' if len(unknowns)>3 else ''})")
            # Use the fixed generation helper, passing config
            _, generated_part = generate_sample(model, tokenizer, tokenizer_inv, config, context=context, gen_tokens=gen_tokens, temperature=temperature)
            print(f"Generated: {generated_part}")
            generated_texts.append({"context": context, "generated": generated_part})
    except FileNotFoundError: print(f"Error: Test file '{test_file}' not found for generation.")
    except Exception as e: print(f"Error during text generation: {e}")

    return generated_texts, test_perplexity

# SimpleNamespace shim for non-notebook environments
try: from argparse import Namespace as SimpleNamespace
except ImportError:
    class SimpleNamespace:
        def __init__(self, **kwargs): self.__dict__.update(kwargs)

# --- Main Execution Guard ---
if __name__ == "__main__":
    main()

Using device: cuda
Loading data files...
Info: Test file 'shakespear_test.txt' not found during initial load.
Building word vocabulary...


Counting words: 100%|██████████| 9837/9837 [00:00<00:00, 83357.99it/s]


Vocabulary size: 5796 (min_freq=2)
Sample vocab: ['<PAD>', '<UNK>', '<START>', '<STOP>', 'first', 'citizen', ':', 'before', 'we', 'proceed'] ... ['wiser', 'hannibal', 'pomphey', 'bum', 'whipt', 'severe', 'isabel', 'giant', 'durance', 'prenzie']
Tokenizing 9837 lines for dataset (max_len=128)...


Tokenizing lines: 100%|██████████| 9837/9837 [00:00<00:00, 21751.70it/s]


Created dataset with 9837 valid sequences.
Tokenizing 1304 lines for dataset (max_len=128)...


...


Iter 24750: Train Loss (batch) 1.2328, PPL 3.43 | Val Loss 6.9966, PPL 1092.92 | LR 0.000030 | Time 1671.6s
 (Best Val PPL remains 86.8609)
Sample Gen:
---
sicinius: we are <UNK> upon a public wood, and have hearts <UNK> to honour and advance the people.
---


Epoch 81 Training: 100%|██████████| 308/308 [00:20<00:00, 14.98it/s, loss=1.0644, PPL=2.9, lr=0.000030]


--- Epoch 81 Finished (20.56s). Total Iters: 24948 ---

--- Starting Epoch 82 ---


Epoch 82 Training:  17%|█▋        | 52/308 [00:04<00:20, 12.31it/s, loss=1.1638, PPL=3.2, lr=0.000030]



Iter 24999: Train Loss (batch) 1.1638, PPL 3.20 | Val Loss 6.9976, PPL 1094.05 | LR 0.000030 | Time 1688.4s
 (Best Val PPL remains 86.8609)
Sample Gen:
---
queen elizabeth: and shall i forget myself?
---
--- Epoch 82 Finished (4.23s). Total Iters: 25000 ---

Training finished.
Total Training Time: 1689.26 seconds
Best Validation Perplexity achieved: 86.8609

Loss/LR plot saved: task1_loss_lr_plot_word_level_improved.png
Perplexity plot saved: task1_loss_lr_plot_word_level_improved_perplexity.png

--- Running Inference on shakespear_test.txt using best model task1_transformer_word_level_improved.pth ---

--- Starting Inference ---
Loading model from task1_transformer_word_level_improved.pth...
TransformerLM (Word-Level Improved) initialized.
 - Vocab Size: 5796, Pad ID: 0
 - Embedding Dim: 384, Block Size (MAX_LEN): 128
 - Layers: 6, Heads: 6
 - Dropout: 0.2, Weight Decay: 0.1
 - Total Params: 12.90 M
Model loaded.

Reading test file for PPL: shakespear_test.txt
Tokenizing 2 lines for 